# LPCMCI simulation baseline

Run this notebook independently of `oasis.ipynb`. Install the optional PAG environment first with `uv sync --extra pag`. Raw PAG tensors are primary; the saved lagged skeleton is a lossy support-only projection.

<!-- reviewer-resume-contract -->
## Execution and resume contract

Each outer-run–condition–seed unit is checkpointed. It recreates the exact static c-GC/c-GC* network and fluorescence input and records an input digest. Re-run the identical cell after interruption; do not change the grid, representations, or output directory while resuming.

In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys

def find_package_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for base in (current, *current.parents):
        for candidate in (base, base / 'calcium-transient-rising-flank'):
            if (candidate / 'pyproject.toml').is_file() and (candidate / 'examples' / 'simulation_baselines.py').is_file():
                return candidate
    raise FileNotFoundError('Could not locate the calcium-transient-rising-flank package root')

PACKAGE_ROOT = find_package_root()
RUNNER_PYTHON = next((str(path) for path in (PACKAGE_ROOT / '.venv/bin/python', PACKAGE_ROOT / '.venv/Scripts/python.exe') if path.is_file()), sys.executable)
RUNNER_ENV = os.environ.copy()
SOURCE_ROOT = str(PACKAGE_ROOT / 'src')
RUNNER_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, (SOURCE_ROOT, RUNNER_ENV.get('PYTHONPATH'))))

RUN_LPCMCI = True
N_RUNS_OUTER = 10
N_SEEDS = 20
N_STEPS = 3000
OUTPUT_DIR = PACKAGE_ROOT / 'outputs/revision_campaign/lpcmci_simulation'
command = [
    RUNNER_PYTHON, 'examples/simulation_baselines.py',
    '--components', 'lpcmci',
    '--representations', 'full,deconvolved,rise,fall,fall_residual',
    '--n-runs-outer', str(N_RUNS_OUTER), '--n-seeds', str(N_SEEDS),
    '--n-steps', str(N_STEPS), '--output-dir', str(OUTPUT_DIR), '--resume',
]
print(shlex.join(command))
if RUN_LPCMCI:
    subprocess.run(command, cwd=PACKAGE_ROOT, env=RUNNER_ENV, check=True)

summary_path = OUTPUT_DIR / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print(json.dumps(summary, indent=2))